In [ ]:
!wget -q "https://www.timeseriesclassification.com/aeon-toolkit/Heartbeat.zip"
!unzip -q Heartbeat.zip

# .ts format needs aeon or sktime to load
!pip install aeon -q

from aeon.datasets import load_from_ts_file
import numpy as np

X_train, y_train = load_from_ts_file("Heartbeat_TRAIN.ts")
X_test,  y_test  = load_from_ts_file("Heartbeat_TEST.ts")

# X comes out as (samples, channels, timesteps) — already correct shape
# Convert to float32
X_train = X_train.astype(np.float32)
X_test  = X_test.astype(np.float32)

# Encode labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(y_train).astype(np.int64)
y_test  = le.transform(y_test).astype(np.int64)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Sequence length: {X_train.shape[2]}")
print(f"Channels: {X_train.shape[1]}")
print(f"Classes: {len(np.unique(y_train))}")

replace Heartbeat.JPG? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import time
from scipy import signal

# Average 61 channels to 1 to maintain architectural consistency
X_train = X_train.mean(axis=1, keepdims=True)  # (204, 1, 405)
X_test  = X_test.mean(axis=1, keepdims=True)   # (205, 1, 405)

# Store original data for resizing later
X_train_original = X_train.copy()
X_test_original = X_test.copy()

# Normalize per sample (for original length 405)
mean = X_train.mean(axis=2, keepdims=True)
std  = X_train.std(axis=2,  keepdims=True) + 1e-8
X_train = (X_train - mean) / std

mean = X_test.mean(axis=2, keepdims=True)
std  = X_test.std(axis=2,  keepdims=True) + 1e-8
X_test = (X_test - mean) / std

X_train_t = torch.tensor(X_train.astype(np.float32))
X_test_t  = torch.tensor(X_test.astype(np.float32))
y_train_t = torch.tensor(y_train)
y_test_t  = torch.tensor(y_test)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=32, shuffle=False)

print(f"Input shape: {X_train_t.shape}")  # should be (204, 1, 405)

# Add resize function here
def resize_sequences(X, target_length):
    """
    Resample sequences to target length using scipy's resample function
    """
    n_samples, channels, orig_length = X.shape
    resized = np.zeros((n_samples, channels, target_length))

    for i in range(n_samples):
        for c in range(channels):
            resized[i, c] = signal.resample(X[i, c], target_length)

    return resized.astype(np.float32)

In [ ]:
class CNN_TemporalPooling(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.network(x)
        x = self.gap(x).squeeze(-1)
        return self.classifier(x)


class CNN_LSTM(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.lstm = nn.LSTM(input_size=128, hidden_size=128,
                            num_layers=2, batch_first=True)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = x.permute(0, 2, 1)
        _, (h_n, _) = self.lstm(x)
        return self.classifier(h_n[-1])

In [ ]:
def train_model_es(model, train_loader, test_loader, epochs=50, lr=1e-3, patience=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_losses, test_accs = [], []
    best_acc = 0.0
    patience_counter = 0
    best_epoch = 0
    start = time.time()

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                pred = model(X_batch.to(device)).argmax(dim=1).cpu().numpy()
                preds.extend(pred)
                labels.extend(y_batch.numpy())

        acc = accuracy_score(labels, preds)
        train_losses.append(epoch_loss / len(train_loader))
        test_accs.append(acc)

        # Early stopping on accuracy
        if acc > best_acc:
            best_acc = acc
            patience_counter = 0
            best_epoch = epoch + 1
            torch.save(model.state_dict(), 'best_model.pt')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}, best epoch was {best_epoch}")
                break

        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1} | Loss: {train_losses[-1]:.4f} | Acc: {acc:.4f}")

    training_time = time.time() - start
    return train_losses, test_accs, training_time, best_acc, best_epoch

In [ ]:
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
import pickle

NUM_CLASSES = len(np.unique(y_train))

set_seed(42)
print("Training CNN + Temporal Pooling...")
pool_model = CNN_TemporalPooling(NUM_CLASSES)
pool_losses, pool_accs, pool_time, pool_best, pool_epoch = train_model_es(
    pool_model, train_loader, test_loader)

set_seed(42)
print("\nTraining CNN + LSTM...")
lstm_model = CNN_LSTM(NUM_CLASSES)
lstm_losses, lstm_accs, lstm_time, lstm_best, lstm_epoch = train_model_es(
    lstm_model, train_loader, test_loader)

# Save immediately
with open('heartbeat_curves.pkl', 'wb') as f:
    pickle.dump({
        'pool_losses': pool_losses, 'lstm_losses': lstm_losses,
        'pool_accs': pool_accs,     'lstm_accs': lstm_accs,
    }, f)

print(f"\n--- Final Results on Heartbeat (T=405) ---")
print(f"CNN+Pooling | Acc: {pool_best:.4f} | Stopped: epoch {pool_epoch} | Time: {pool_time:.1f}s")
print(f"CNN+LSTM    | Acc: {lstm_best:.4f} | Stopped: epoch {lstm_epoch} | Time: {lstm_time:.1f}s")

In [ ]:
# ==================== TEST DIFFERENT SEQUENCE LENGTHS ====================
def test_multiple_lengths():
    """Test models on different sequence lengths"""

    lengths_to_test = [160, 240, 360, 405]
    results = {}
    NUM_CLASSES = len(np.unique(y_train))

    for seq_len in lengths_to_test:
        print(f"\n{'='*60}")
        print(f"Testing sequence length: {seq_len}")
        print(f"{'='*60}")

        # Resize sequences if needed
        if seq_len != X_train_original.shape[2]:
            print(f"Resizing from {X_train_original.shape[2]} to {seq_len}...")
            X_train_resized = resize_sequences(X_train_original, seq_len)
            X_test_resized = resize_sequences(X_test_original, seq_len)
        else:
            print("Using original length...")
            X_train_resized = X_train_original
            X_test_resized = X_test_original

        # Normalize
        mean = X_train_resized.mean(axis=2, keepdims=True)
        std = X_train_resized.std(axis=2, keepdims=True) + 1e-8
        X_train_norm = (X_train_resized - mean) / std

        mean = X_test_resized.mean(axis=2, keepdims=True)
        std = X_test_resized.std(axis=2, keepdims=True) + 1e-8
        X_test_norm = (X_test_resized - mean) / std

        # Convert to tensors
        X_train_t = torch.tensor(X_train_norm.astype(np.float32))
        X_test_t = torch.tensor(X_test_norm.astype(np.float32))
        y_train_t = torch.tensor(y_train)
        y_test_t = torch.tensor(y_test)

        print(f"Resized shape - Train: {X_train_t.shape}, Test: {X_test_t.shape}")

        # Create loaders
        train_loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                                batch_size=32, shuffle=True)
        test_loader = DataLoader(TensorDataset(X_test_t, y_test_t),
                               batch_size=32, shuffle=False)

        # Train CNN + Temporal Pooling
        set_seed(42)
        print(f"\nTraining CNN + Temporal Pooling for length {seq_len}...")
        pool_model = CNN_TemporalPooling(NUM_CLASSES)
        pool_losses, pool_accs, pool_time, pool_best, pool_epoch = train_model_es(
            pool_model, train_loader, test_loader)

        # Train CNN + LSTM
        set_seed(42)
        print(f"\nTraining CNN + LSTM for length {seq_len}...")
        lstm_model = CNN_LSTM(NUM_CLASSES)
        lstm_losses, lstm_accs, lstm_time, lstm_best, lstm_epoch = train_model_es(
            lstm_model, train_loader, test_loader)

        # Store results
        results[seq_len] = {
            'pool': {
                'best_acc': pool_best,
                'time': pool_time,
                'epoch': pool_epoch,
                'losses': pool_losses,
                'accs': pool_accs
            },
            'lstm': {
                'best_acc': lstm_best,
                'time': lstm_time,
                'epoch': lstm_epoch,
                'losses': lstm_losses,
                'accs': lstm_accs
            }
        }

        print(f"\nResults for length {seq_len}:")
        print(f"CNN+Pooling | Acc: {pool_best:.4f} | Best Epoch: {pool_epoch} | Time: {pool_time:.1f}s")
        print(f"CNN+LSTM    | Acc: {lstm_best:.4f} | Best Epoch: {lstm_epoch} | Time: {lstm_time:.1f}s")

    return results

# Run the multi-length test
print("\n" + "="*60)
print("STARTING MULTI-LENGTH EVALUATION")
print("="*60)

multi_results = test_multiple_lengths()

# Save results
with open('heartbeat_length_comparison.pkl', 'wb') as f:
    pickle.dump(multi_results, f)
print("\nResults saved to 'heartbeat_length_comparison.pkl'")

# Print summary table
print("\n" + "="*80)
print(f"{'Length':<10} {'CNN+Pooling Acc':<20} {'CNN+LSTM Acc':<20} {'Pooling Time':<15} {'LSTM Time':<15}")
print("-"*80)
for l in [160, 240, 360, 405]:
    print(f"{l:<10} {multi_results[l]['pool']['best_acc']:<20.4f} {multi_results[l]['lstm']['best_acc']:<20.4f} "
          f"{multi_results[l]['pool']['time']:<15.1f} {multi_results[l]['lstm']['time']:<15.1f}")
print("="*80)

In [ ]:
import matplotlib.pyplot as plt

# Create comparison plots for all lengths
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

lengths = [160, 240, 360, 405]

# 1. Accuracy vs Length
pool_accs = [multi_results[l]['pool']['best_acc'] for l in lengths]
lstm_accs = [multi_results[l]['lstm']['best_acc'] for l in lengths]

axes[0, 0].plot(lengths, pool_accs, 'o-', label='CNN+Pooling', linewidth=2, markersize=10)
axes[0, 0].plot(lengths, lstm_accs, 's-', label='CNN+LSTM', linewidth=2, markersize=10)
axes[0, 0].set_xlabel('Sequence Length', fontsize=12)
axes[0, 0].set_ylabel('Test Accuracy', fontsize=12)
axes[0, 0].set_title('Accuracy vs Sequence Length', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=12)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_ylim([0, 1])

# 2. Training Time vs Length
pool_times = [multi_results[l]['pool']['time'] for l in lengths]
lstm_times = [multi_results[l]['lstm']['time'] for l in lengths]

axes[0, 1].plot(lengths, pool_times, 'o-', label='CNN+Pooling', linewidth=2, markersize=10)
axes[0, 1].plot(lengths, lstm_times, 's-', label='CNN+LSTM', linewidth=2, markersize=10)
axes[0, 1].set_xlabel('Sequence Length', fontsize=12)
axes[0, 1].set_ylabel('Training Time (seconds)', fontsize=12)
axes[0, 1].set_title('Training Time vs Sequence Length', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=12)
axes[0, 1].grid(True, alpha=0.3)

# 3. Accuracy curves for all lengths (CNN+Pooling)
for l in lengths:
    axes[1, 0].plot(multi_results[l]['pool']['accs'],
                   label=f'Length {l}', linewidth=2)
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Test Accuracy', fontsize=12)
axes[1, 0].set_title('CNN+Pooling: Accuracy Curves by Length', fontsize=14, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# 4. Accuracy curves for all lengths (CNN+LSTM)
for l in lengths:
    axes[1, 1].plot(multi_results[l]['lstm']['accs'],
                   label=f'Length {l}', linewidth=2)
axes[1, 1].set_xlabel('Epoch', fontsize=12)
axes[1, 1].set_ylabel('Test Accuracy', fontsize=12)
axes[1, 1].set_title('CNN+LSTM: Accuracy Curves by Length', fontsize=14, fontweight='bold')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('heartbeat_length_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Multi-length evaluation complete!")
print("Check the plots above and 'heartbeat_length_comparison.png'")